# Merging, Joining & Concatenating — combining two tables into one

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`02_Pandas_Essentials/04_merging_joining_concatenating.ipynb`

---

### In one paragraph (no jargon)

Real data arrives in pieces: customers in one file, their orders in another, product details in a third. Combining them is two distinct jobs and confusing them is the commonest mistake. **Concatenating** stacks tables that have the same shape — January's sales on top of February's. **Merging** matches rows across tables using a shared key — attaching each order to the customer who placed it. Excel users know merging as VLOOKUP; SQL users know it as JOIN. Same idea, four flavours.

### After this notebook you can

- Stack tables vertically and side by side with `pd.concat`
- Match two tables on a shared key with `pd.merge`
- Choose correctly between inner, left, right and outer joins — and predict the row count
- Spot and diagnose the two classic merge disasters: silent row loss and accidental duplication

**Assumed knowledge:** `01_series_and_dataframes.ipynb`

### What's inside

1. Concatenating — stacking tables
2. Merging — matching on a key
3. The four join types, and how to choose
4. Joining on the index with .join()
5. ⚡ Validating a merge before it silently ruins your analysis
6. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    return rebuild()

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


In [2]:
# Three months of sales, same columns — the classic case for CONCATENATION
jan = pd.DataFrame({'Order_ID': ['A0', 'A1', 'A2', 'A3'],
                    'Customer': ['C0', 'C1', 'C2', 'C3'],
                    'Amount':   [1200, 900, 1500, 800]}, index=[0, 1, 2, 3])
feb = pd.DataFrame({'Order_ID': ['A4', 'A5', 'A6', 'A7'],
                    'Customer': ['C4', 'C5', 'C6', 'C7'],
                    'Amount':   [1100, 1750, 640, 2200]}, index=[4, 5, 6, 7])
mar = pd.DataFrame({'Order_ID': ['A8', 'A9', 'A10', 'A11'],
                    'Customer': ['C8', 'C9', 'C10', 'C11'],
                    'Amount':   [980, 1320, 1490, 700]}, index=[8, 9, 10, 11])

print("January:"); display(jan)
print("February:"); display(feb)

January:


,Order_ID,Customer,Amount
0,A0,C0,1200
1,A1,C1,900
2,A2,C2,1500
3,A3,C3,800


February:


,Order_ID,Customer,Amount
4,A4,C4,1100
5,A5,C5,1750
6,A6,C6,640
7,A7,C7,2200


## 1. Concatenating — stacking tables

`pd.concat([a, b, c])` glues tables together. The `axis` argument decides the direction:

- `axis=0` (the default) stacks them **vertically** — more rows. This is what you want 95% of the time.
- `axis=1` places them **side by side** — more columns. It aligns on the **index**, not on any key, so it is
  only safe when both tables describe the same rows in the same order.

In [3]:
# axis=0 — vertical stacking, the normal case
full_quarter = pd.concat([jan, feb, mar])
print(f"Three tables of {len(jan)} rows -> one table of {len(full_quarter)} rows")
display(full_quarter)

# ignore_index=True renumbers 0..n-1. Do this whenever the original indexes are meaningless.
print("With ignore_index=True the row labels are renumbered:")
display(pd.concat([jan, feb, mar], ignore_index=True).tail(3))

# keys= labels each source — invaluable for tracking where a row came from
labelled = pd.concat([jan, feb, mar], keys=['Jan', 'Feb', 'Mar'])
print("\nWith keys=, the source becomes part of the index:")
display(labelled.head(6))
print("\nWhich makes 'total by month' immediate:")
print(labelled.groupby(level=0)['Amount'].sum().to_string())

Three tables of 4 rows -> one table of 12 rows


,Order_ID,Customer,Amount
0,A0,C0,1200
1,A1,C1,900
2,A2,C2,1500
3,A3,C3,800
4,A4,C4,1100
5,A5,C5,1750
6,A6,C6,640
7,A7,C7,2200
8,A8,C8,980
9,A9,C9,1320


With ignore_index=True the row labels are renumbered:


,Order_ID,Customer,Amount
9,A9,C9,1320
10,A10,C10,1490
11,A11,C11,700



With keys=, the source becomes part of the index:


Order_ID Customer  Amount
Jan 0       A0       C0    1200
    1       A1       C1     900
    2       A2       C2    1500
    3       A3       C3     800
Feb 4       A4       C4    1100
    5       A5       C5    1750


Which makes 'total by month' immediate:
Feb    5690
Jan    4400
Mar    4490


In [4]:
# axis=1 — side by side. Aligns on the INDEX, which is where it goes wrong.
print("axis=1 with matching indexes — fine:")
display(pd.concat([jan[['Order_ID']], jan[['Amount']]], axis=1))

print("\naxis=1 with DIFFERENT indexes — pandas fills the gaps with NaN:")
display(pd.concat([jan, feb], axis=1).head(5))
print("^ jan has indexes 0-3 and feb has 4-7, so nothing lines up and you get 8 half-empty rows.")
print("  If you meant to match on a key rather than on row position, you want pd.merge — see below.")

# Columns that don't match are unioned, with NaN where a table lacked the column
apr = pd.DataFrame({'Order_ID': ['A12'], 'Customer': ['C12'], 'Amount': [500], 'Channel': ['Online']})
print("\nStacking tables with different columns:")
display(pd.concat([jan.head(2), apr], ignore_index=True))
print("^ 'Channel' is NaN for the January rows because that column didn't exist then. Usually fine —")
print("  but if you EXPECTED matching columns, this is your warning that something's off.")

axis=1 with matching indexes — fine:


,Order_ID,Amount
0,A0,1200
1,A1,900
2,A2,1500
3,A3,800



axis=1 with DIFFERENT indexes — pandas fills the gaps with NaN:


,Order_ID,Customer,Amount,Order_ID,Customer,Amount
0,A0,C0,1200.0,NaN,NaN,NaN
1,A1,C1,900.0,NaN,NaN,NaN
2,A2,C2,1500.0,NaN,NaN,NaN
3,A3,C3,800.0,NaN,NaN,NaN
4,NaN,NaN,NaN,A4,C4,1100.0


^ jan has indexes 0-3 and feb has 4-7, so nothing lines up and you get 8 half-empty rows.
  If you meant to match on a key rather than on row position, you want pd.merge — see below.

Stacking tables with different columns:


,Order_ID,Customer,Amount,Channel
0,A0,C0,1200,NaN
1,A1,C1,900,NaN
2,A12,C12,500,Online


^ 'Channel' is NaN for the January rows because that column didn't exist then. Usually fine —
  but if you EXPECTED matching columns, this is your warning that something's off.


## 2. Merging — matching on a key

`pd.merge(left, right, how=..., on='key')` matches rows between two tables wherever the key column agrees.
This is the VLOOKUP/JOIN operation.

The four `how=` values differ only in **which non-matching rows survive**:

| `how=` | Keeps | Use when |
|---|---|---|
| `'inner'` (default) | only keys present in **both** | you need complete records |
| `'left'` | **all** left rows, plus matches | enriching a master list — **the safest default** |
| `'right'` | all right rows, plus matches | rare; just swap the tables and use `left` |
| `'outer'` | everything from both | reconciling two sources, finding what's missing from each |

In [5]:
customers_tbl = pd.DataFrame({
    'Customer': ['C0', 'C1', 'C2', 'C3'],
    'Name':     ['Ravi', 'Meera', 'John', 'Ali'],
    'City':     ['Delhi', 'Mumbai', 'Pune', 'Delhi'],
})
orders_tbl = pd.DataFrame({
    'Customer': ['C0', 'C1', 'C2', 'C9'],       # C9 has no matching customer record
    'Order_ID': ['A0', 'A1', 'A2', 'A9'],
    'Amount':   [1200, 900, 1500, 4000],
})
print("Customers:"); display(customers_tbl)
print("Orders (note C9 — an order with no customer record):"); display(orders_tbl)

print("\nINNER — only customers who have orders AND orders that have customers:")
display(pd.merge(customers_tbl, orders_tbl, how='inner', on='Customer'))
print("^ C3 (no orders) and C9 (no customer record) are both gone. 4+4 rows became 3.")

Customers:


,Customer,Name,City
0,C0,Ravi,Delhi
1,C1,Meera,Mumbai
2,C2,John,Pune
3,C3,Ali,Delhi


Orders (note C9 — an order with no customer record):


,Customer,Order_ID,Amount
0,C0,A0,1200
1,C1,A1,900
2,C2,A2,1500
3,C9,A9,4000



INNER — only customers who have orders AND orders that have customers:


,Customer,Name,City,Order_ID,Amount
0,C0,Ravi,Delhi,A0,1200
1,C1,Meera,Mumbai,A1,900
2,C2,John,Pune,A2,1500


^ C3 (no orders) and C9 (no customer record) are both gone. 4+4 rows became 3.


In [6]:
print("LEFT — every customer kept; those without orders get NaN:")
display(pd.merge(customers_tbl, orders_tbl, how='left', on='Customer'))
print("^ C3 survives with NaN order details. This is usually what you want.\n")

print("RIGHT — every order kept; the orphan order C9 gets NaN customer details:")
display(pd.merge(customers_tbl, orders_tbl, how='right', on='Customer'))

print("\nOUTER — everything from both sides:")
display(pd.merge(customers_tbl, orders_tbl, how='outer', on='Customer'))
print("^ Both C3 and C9 appear. Use outer when you need to SEE what doesn't match.")

LEFT — every customer kept; those without orders get NaN:


,Customer,Name,City,Order_ID,Amount
0,C0,Ravi,Delhi,A0,1200.0
1,C1,Meera,Mumbai,A1,900.0
2,C2,John,Pune,A2,1500.0
3,C3,Ali,Delhi,NaN,NaN


^ C3 survives with NaN order details. This is usually what you want.

RIGHT — every order kept; the orphan order C9 gets NaN customer details:


,Customer,Name,City,Order_ID,Amount
0,C0,Ravi,Delhi,A0,1200
1,C1,Meera,Mumbai,A1,900
2,C2,John,Pune,A2,1500
3,C9,NaN,NaN,A9,4000



OUTER — everything from both sides:


,Customer,Name,City,Order_ID,Amount
0,C0,Ravi,Delhi,A0,1200.0
1,C1,Meera,Mumbai,A1,900.0
2,C2,John,Pune,A2,1500.0
3,C3,Ali,Delhi,NaN,NaN
4,C9,NaN,NaN,A9,4000.0


^ Both C3 and C9 appear. Use outer when you need to SEE what doesn't match.


In [7]:
# Merging on SEVERAL keys — pass a list. Rows match only when every key agrees.
left = pd.DataFrame({'region': ['N', 'N', 'S', 'W'],
                     'quarter': ['Q1', 'Q2', 'Q1', 'Q2'],
                     'target': [100, 120, 90, 110]})
right = pd.DataFrame({'region': ['N', 'N', 'S', 'E'],
                      'quarter': ['Q1', 'Q2', 'Q1', 'Q1'],
                      'actual': [104, 111, 95, 80]})

print("Merged on region AND quarter:")
merged = pd.merge(left, right, how='outer', on=['region', 'quarter'])
display(merged)

# Differently-named key columns: left_on / right_on
a = pd.DataFrame({'cust_code': ['C0', 'C1'], 'name': ['Ravi', 'Meera']})
b = pd.DataFrame({'customer_id': ['C0', 'C1'], 'spend': [1200, 900]})
print("\nKeys with different names — left_on / right_on:")
display(pd.merge(a, b, left_on='cust_code', right_on='customer_id', how='inner'))

# Same column name on both sides that ISN'T a key -> pandas adds _x and _y. Rename them.
p = pd.DataFrame({'id': [1, 2], 'value': [10, 20]})
q = pd.DataFrame({'id': [1, 2], 'value': [99, 88]})
print("\nClashing non-key column names:")
display(pd.merge(p, q, on='id'))
print("Fix with suffixes= so the columns say what they are:")
display(pd.merge(p, q, on='id', suffixes=('_budget', '_actual')))

Merged on region AND quarter:


,region,quarter,target,actual
0,E,Q1,NaN,80.0
1,N,Q1,100.0,104.0
2,N,Q2,120.0,111.0
3,S,Q1,90.0,95.0
4,W,Q2,110.0,NaN



Keys with different names — left_on / right_on:


,cust_code,name,customer_id,spend
0,C0,Ravi,C0,1200
1,C1,Meera,C1,900



Clashing non-key column names:


,id,value_x,value_y
0,1,10,99
1,2,20,88


Fix with suffixes= so the columns say what they are:


,id,value_budget,value_actual
0,1,10,99
1,2,20,88


## 3. Joining on the index with `.join()`

`.join()` is `merge`'s convenience wrapper for when the key **is the index** of both tables. It defaults to a
*left* join (unlike `merge`, which defaults to inner) — a small inconsistency that catches people out.

In practice: use `pd.merge` for keys held in columns, `.join()` for keys held in the index.

In [8]:
left_idx = pd.DataFrame({'A': ['A0', 'A1', 'A2'], 'B': ['B0', 'B1', 'B2']},
                        index=['K0', 'K1', 'K2'])
right_idx = pd.DataFrame({'C': ['C0', 'C2', 'C3'], 'D': ['D0', 'D2', 'D3']},
                         index=['K0', 'K2', 'K3'])
print("left_idx:"); display(left_idx)
print("right_idx:"); display(right_idx)

print("\n.join() — defaults to LEFT (note: merge defaults to inner!):")
display(left_idx.join(right_idx))

print(".join(how='outer') — every key from both:")
display(left_idx.join(right_idx, how='outer'))

print("\nThe merge equivalent, spelled out:")
display(pd.merge(left_idx, right_idx, left_index=True, right_index=True, how='left'))

left_idx:

,A,B
K0,A0,B0
K1,A1,B1
K2,A2,B2


right_idx:


,C,D
K0,C0,D0
K2,C2,D2
K3,C3,D3



.join() — defaults to LEFT (note: merge defaults to inner!):


,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2


.join(how='outer') — every key from both:


,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2
K3,NaN,NaN,C3,D3



The merge equivalent, spelled out:


,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2


### ⚡ Beyond the syllabus — validating a merge before it silently ruins your analysis

Merges fail **quietly**. The two disasters: rows vanish because keys didn't match (whitespace, case, `'01'` vs `1`), or rows multiply because the right-hand key wasn't unique and every left row matched several. Neither raises an error. The `validate=` argument and the `indicator=` column turn both into something you can see — this single habit prevents more wrong answers than any other.

In [9]:
# ---- DISASTER 1: silent row loss from mismatched key formats -----------------
left_bad  = pd.DataFrame({'cust': ['C0', 'C1 ', 'c2'], 'name': ['Ravi', 'Meera', 'John']})
right_bad = pd.DataFrame({'cust': ['C0', 'C1', 'C2'],  'spend': [1200, 900, 1500]})

naive = pd.merge(left_bad, right_bad, on='cust', how='inner')
print(f"Expected 3 matches, actually got {len(naive)} — trailing space and lowercase broke two of them:")
display(naive)

# indicator=True adds a _merge column showing where each row came from
audit = pd.merge(left_bad, right_bad, on='cust', how='outer', indicator=True)
print("\nWith indicator=True — the diagnosis is visible:")
display(audit)
print("\nUnmatched summary:", audit['_merge'].value_counts().to_dict())

# THE FIX: normalise the key on both sides before merging
fixed = pd.merge(left_bad.assign(cust=left_bad['cust'].str.strip().str.upper()),
                 right_bad.assign(cust=right_bad['cust'].str.strip().str.upper()),
                 on='cust', how='inner')
print(f"\nAfter stripping whitespace and standardising case: {len(fixed)} matches")
display(fixed)

Expected 3 matches, actually got 1 — trailing space and lowercase broke two of them:


,cust,name,spend
0,C0,Ravi,1200



With indicator=True — the diagnosis is visible:


,cust,name,spend,_merge
0,C0,Ravi,1200.0,both
1,C1,NaN,900.0,right_only
2,C1,Meera,NaN,left_only
3,C2,NaN,1500.0,right_only
4,c2,John,NaN,left_only



Unmatched summary: {'left_only': 2, 'right_only': 2, 'both': 1}

After stripping whitespace and standardising case: 3 matches


,cust,name,spend
0,C0,Ravi,1200
1,C1,Meera,900
2,C2,John,1500


In [10]:
# ---- DISASTER 2: silent row multiplication -----------------------------------
orders_l = pd.DataFrame({'cust': ['C0', 'C1'], 'order': ['A0', 'A1']})
lookup_r = pd.DataFrame({'cust': ['C0', 'C0', 'C1'],           # C0 appears TWICE
                         'city': ['Delhi', 'Delhi (old)', 'Mumbai']})

blown_up = pd.merge(orders_l, lookup_r, on='cust', how='left')
print(f"2 orders merged with a lookup -> {len(blown_up)} rows. Every total is now overstated:")
display(blown_up)

# validate= makes pandas REFUSE rather than quietly duplicate
print("\nvalidate='one_to_one' catches it immediately:")
try:
    pd.merge(orders_l, lookup_r, on='cust', how='left', validate='one_to_one')
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

print("""
  validate= options
    'one_to_one'   keys unique on BOTH sides
    'one_to_many'  left unique, right may repeat   <- the usual master-to-detail case
    'many_to_one'  right unique, left may repeat   <- the usual lookup case
""")

# THE FIX: de-duplicate the lookup table first, then merge
clean_lookup = lookup_r.drop_duplicates(subset='cust', keep='first')
safe = pd.merge(orders_l, clean_lookup, on='cust', how='left', validate='many_to_one')
print(f"After de-duplicating the lookup: {len(safe)} rows — correct.")
display(safe)

2 orders merged with a lookup -> 3 rows. Every total is now overstated:


,cust,order,city
0,C0,A0,Delhi
1,C0,A0,Delhi (old)
2,C1,A1,Mumbai



validate='one_to_one' catches it immediately:
  MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 cust
  C0 ...

  validate= options
    'one_to_one'   keys unique on BOTH sides
    'one_to_many'  left unique, right may repeat   <- the usual master-to-detail case
    'many_to_one'  right unique, left may repeat   <- the usual lookup case



After de-duplicating the lookup: 2 rows — correct.


,cust,order,city
0,C0,A0,Delhi
1,C1,A1,Mumbai


In [11]:
# ---- The habit worth building: check the row count every single time --------
def merge_report(left, right, **kwargs):
    """Merge, then say plainly what happened to the rows."""
    out = pd.merge(left, right, indicator=True, **kwargs)
    counts = out['_merge'].value_counts()
    print(f"  left rows in : {len(left):>5}")
    print(f"  right rows in: {len(right):>5}")
    print(f"  rows out     : {len(out):>5}")
    print(f"  matched both : {counts.get('both', 0):>5}")
    print(f"  left only    : {counts.get('left_only', 0):>5}   <- lost if you use how='inner'")
    print(f"  right only   : {counts.get('right_only', 0):>5}")
    if len(out) > max(len(left), len(right)):
        print("  ⚠ ROW COUNT GREW — your key is not unique on one side. Check before proceeding.")
    return out.drop(columns='_merge')

print("Merging customers with orders:")
result = merge_report(customers_tbl, orders_tbl, on='Customer', how='outer')
display(result)

Merging customers with orders:

  left rows in :     4
  right rows in:     4
  rows out     :     5
  matched both :     3
  left only    :     1   <- lost if you use how='inner'
  right only   :     1
  ⚠ ROW COUNT GREW — your key is not unique on one side. Check before proceeding.


,Customer,Name,City,Order_ID,Amount
0,C0,Ravi,Delhi,A0,1200.0
1,C1,Meera,Mumbai,A1,900.0
2,C2,John,Pune,A2,1500.0
3,C3,Ali,Delhi,NaN,NaN
4,C9,NaN,NaN,A9,4000.0


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Stack vertically | `pd.concat([a, b])` |
| Stack and renumber | `pd.concat([a, b], ignore_index=True)` |
| Tag the source | `pd.concat([a, b], keys=['Jan','Feb'])` |
| Side by side | `pd.concat([a, b], axis=1)` |
| Match on a key | `pd.merge(a, b, on='key')` |
| Keep all left rows | `pd.merge(a, b, on='key', how='left')` |
| Keep everything | `… how='outer'` |
| Several keys | `on=['region', 'quarter']` |
| Differently-named keys | `left_on='a_id', right_on='b_id'` |
| Rename clashing columns | `suffixes=('_budget', '_actual')` |
| Join on the index | `a.join(b)` |
| Show what matched | `indicator=True` |
| Refuse a bad merge | `validate='one_to_many'` |
| Find unmatched rows | `out[out['_merge'] != 'both']` |

### Adapting this in the exam

- Same columns, more rows → `concat`. Different columns, matched by a key → `merge`.
- 'Include all customers even without orders' → `how='left'` with customers on the left.
- 'Only customers who ordered' → `how='inner'`.
- 'Which customers have no orders?' → `how='left'` with `indicator=True`, then filter `_merge == 'left_only'`.

### Traps that cost marks

- `merge` defaults to **inner** (drops non-matches); `.join()` defaults to **left**. Always state `how=` explicitly.
- Row count *fell* after a merge → keys didn't match. Check for whitespace, case, and text-vs-number types.
- Row count *grew* after a merge → the right-hand key repeats. De-duplicate the lookup table first.
- `concat(axis=1)` aligns on the **index**, not on a key. If you meant to match on a column, use `merge`.
- Merging `'01'` (text) with `1` (number) matches nothing and raises no error. Check `df.dtypes` first.
- Columns with the same name on both sides silently become `_x` and `_y`. Set `suffixes=` to something meaningful.